[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/360760-gold-btc-lazy-factor.ipynb)

# Optimizing Investment Strategies with Lazy Factor and Probability Weighting (2023) — Praxis reproduction

**Shuo Han, Yinan Chen & Jiacheng Liu** · [arXiv 2306.07928](https://arxiv.org/abs/2306.07928) · gold + Bitcoin, 2016–2021

**The claim:** $1,000 grows to **$136,216** (136×) over five years with a **maximum retracement of 2.5%**, by blending ARIMA price forecasts with a Monte-Carlo mean-variance optimizer, anchored 70% to an equal-weight cash/gold/Bitcoin mix (the "lazy factor").

**Our verdict — FAILS:** the claim is arithmetically impossible under the paper's own rule. This notebook re-runs the stated strategy on the paper's own data sources (LBMA gold PM fix, Blockchain.com Bitcoin market price — both public), with strictly no look-ahead, and prints **claimed → measured**. It also runs the diagnostic that settles the question: even handing the optimizer *tomorrow's exact prices*, the stated rule cannot get anywhere near the claimed wealth.

*Same public data and logic as the Praxis engine — this lands on our exact published numbers. The full C0–C7 gate suite (deflation, crisis windows, cost sweeps, post-publication decay, factor spanning) lives in the report: [market-graphs.com/praxis/papers/360760](https://market-graphs.com/praxis/papers/360760).*

Runtime: ~4 minutes (the ARIMA re-fits daily, honestly).

In [ ]:
# Pinned so the ARIMA fits and Monte-Carlo stream reproduce our engine bit-for-bit.
%pip install -q numpy==2.1.2 pandas==2.2.3 statsmodels==0.14.6

In [ ]:
"""Praxis reproduction — paper 360760 (arXiv 2306.07928), gold/BTC lazy-factor.

Self-contained: fetches the paper's own data sources from their public mirrors
(LBMA gold PM fix from the LBMA itself; Bitcoin market price from
Blockchain.com — the series behind NASDAQ Data Link's BCHAIN/MKPRU), then runs
the paper's stated rule with no look-ahead:

  ARIMA(2,1,2) next-price forecasts -> Monte-Carlo max-Sharpe tangency on the
  cash/gold/BTC simplex (costs in the objective) -> lazy blend
  w = 0.7*(1/3,1/3,1/3) + 0.3*ideal -> daily unit-based accounting, $1,000
  start, gold frozen on non-LBMA days. Commissions 1% gold / 2% BTC.

Same data + same logic as the Praxis engine => the exact published numbers.
"""
import json
import urllib.request
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

START, FIRST_DECISION, PAPER_END = "2016-09-11", "2016-10-11", "2021-09-10"
DATA_END = "2021-09-17"     # panel extends past the sample end so the look-ahead
                            # diagnostic can "see tomorrow" on the final day too
                            # (mirrors the engine, which loads through 2026)
COST_GOLD, COST_BTC = 0.01, 0.02          # MCM-2022-C commissions (paper omits its rates)
DELTA_IDEAL, MC_DRAWS, SEED = 0.3, 10_000, 0
EW3, EW2 = np.array([1 / 3, 1 / 3, 1 / 3]), np.array([1 / 2, 1 / 2])
HDR = {"User-Agent": "Mozilla/5.0 (praxis-reproduction)"}

CLAIMED = {"final": 136_216.1, "mdd_pct": -2.5044, "ann_pct": 27.04}


def fetch(url):
    return urllib.request.urlopen(urllib.request.Request(url, headers=HDR), timeout=90).read()


def load_panel(start=START, end=DATA_END):
    raw = json.loads(fetch("https://prices.lbma.org.uk/json/gold_pm.json"))
    gold_fix = pd.Series({pd.Timestamp(r["d"]): r["v"][0] for r in raw
                          if r.get("v") and r["v"][0]}).sort_index().loc[start:end]
    raw = json.loads(fetch("https://api.blockchain.info/charts/market-price"
                           "?timespan=all&format=json&sampled=false"))
    btc = pd.Series({pd.Timestamp(p["x"], unit="s"): p["y"]
                     for p in raw["values"]}).sort_index()
    btc = btc[btc > 0].loc[start:end]
    cal = pd.date_range(start, end, freq="D")
    return {"calendar": cal,
            "gold": gold_fix.reindex(cal).ffill(),
            "gold_traded": pd.Series(cal.isin(gold_fix.index), index=cal),
            "btc": btc.reindex(cal).ffill()}


def arima_next(values, order=(2, 1, 2)):
    from statsmodels.tsa.arima.model import ARIMA
    for o in (order, (1, 1, 1)):
        try:
            fc = float(ARIMA(values, order=o).fit(
                method_kwargs={"maxiter": 50}).forecast(1)[0])
            if np.isfinite(fc) and fc > 0:
                return fc
        except Exception:
            continue
    return float(values[-1])


def build_forecasts(panel, first_decision=FIRST_DECISION, end=PAPER_END, max_obs=100):
    cal = panel["calendar"]
    days = cal[(cal >= pd.Timestamp(first_decision)) & (cal <= pd.Timestamp(end))]
    gold_fix = panel["gold"][panel["gold_traded"]]
    rows = {}
    for i, t in enumerate(days):
        g_hist = gold_fix.loc[:t].values[-max_obs:]
        b_hist = panel["btc"].loc[:t].values[-max_obs:]
        g_fc = arima_next(g_hist) if panel["gold_traded"][t] and len(g_hist) >= 20 else np.nan
        b_fc = arima_next(b_hist) if len(b_hist) >= 20 else np.nan
        rows[t] = (g_fc, b_fc)
        if i % 200 == 0:
            print(f"  forecasts {t.date()} ({i}/{len(days)})")
    return pd.DataFrame.from_dict(rows, orient="index", columns=["gold_fc", "btc_fc"])


def mc_tangency(mu, cov, w_cur, costs, n_draws, rng):
    W = rng.dirichlet(np.ones(len(mu)), size=n_draws)
    net = W @ mu - np.abs(W - w_cur) @ costs
    if net.max() <= 0:
        return W[int(np.argmax(net))]
    sig = np.sqrt(np.einsum("ij,jk,ik->i", W, cov, W).clip(min=0.0))
    score = np.where(net > 0, net / (sig + 1e-9), -np.inf)
    return W[int(np.argmax(score))]


def run_strategy(panel, forecasts, delta_ideal=DELTA_IDEAL, n_draws=MC_DRAWS,
                 cost_gold=COST_GOLD, cost_btc=COST_BTC, cov_lookback=60,
                 w0=1000.0, seed=SEED, foresight=False):
    cal = panel["calendar"]
    gold, gold_traded, btc = panel["gold"], panel["gold_traded"], panel["btc"]
    lg, lb = np.log(gold).diff(), np.log(btc).diff()
    ret2 = pd.concat([lg.rename("g"), lb.rename("b")], axis=1).dropna()
    if foresight:
        g_next, b_next = gold[gold_traded].shift(-1), btc.shift(-1)
    rng = np.random.default_rng(seed)
    C, G, B = float(w0), 0.0, 0.0
    first_dec = pd.Timestamp(FIRST_DECISION)
    wealth, weights = {}, {}
    for t in cal[cal <= pd.Timestamp(PAPER_END)]:
        p_g, p_b = float(gold[t]), float(btc[t])
        W = C + G * p_g + B * p_b
        if t < first_dec:
            wealth[t] = W
            continue
        if foresight:
            g_fc, b_fc = float(g_next.get(t, np.nan)), float(b_next.get(t, np.nan))
        else:
            g_fc = float(forecasts["gold_fc"].get(t, np.nan))
            b_fc = float(forecasts["btc_fc"].get(t, np.nan))
        win = ret2.loc[t - pd.Timedelta(days=cov_lookback):t]
        if gold_traded[t] and np.isfinite(g_fc) and np.isfinite(b_fc) and len(win) >= 20:
            mu = np.array([0.0, np.log(g_fc / p_g), np.log(b_fc / p_b)])
            cov = np.zeros((3, 3)); cov[1:, 1:] = np.cov(win.values, rowvar=False, ddof=1)
            w_cur = np.array([C / W, G * p_g / W, B * p_b / W])
            ideal = mc_tangency(mu, cov, w_cur, np.array([0.0, cost_gold, cost_btc]), n_draws, rng)
            w_tgt = (1 - delta_ideal) * EW3 + delta_ideal * ideal
            G_new, B_new = w_tgt[1] * W / p_g, w_tgt[2] * W / p_b
            cost = cost_gold * abs(G_new - G) * p_g + cost_btc * abs(B_new - B) * p_b
            C, G, B = w_tgt[0] * W - cost, G_new, B_new
        elif np.isfinite(b_fc) and len(win) >= 20:
            Ws = C + B * p_b
            mu = np.array([0.0, np.log(b_fc / p_b)])
            cov = np.zeros((2, 2)); cov[1, 1] = float(win["b"].var(ddof=1))
            ideal = mc_tangency(mu, cov, np.array([C / Ws, B * p_b / Ws]),
                                np.array([0.0, cost_btc]), n_draws, rng)
            w_tgt = (1 - delta_ideal) * EW2 + delta_ideal * ideal
            B_new = w_tgt[1] * Ws / p_b
            cost = cost_btc * abs(B_new - B) * p_b
            C, B = w_tgt[0] * Ws - cost, B_new
        wn = C + G * p_g + B * p_b
        wealth[t] = wn
        weights[t] = (C / wn, G * p_g / wn, B * p_b / wn)
    return pd.Series(wealth).sort_index()


def run_constant_mix(panel, w_fix, cost_gold=COST_GOLD, cost_btc=COST_BTC, w0=1000.0):
    cal = panel["calendar"]
    gold, gold_traded, btc = panel["gold"], panel["gold_traded"], panel["btc"]
    C, G, B = float(w0), 0.0, 0.0
    first_dec = pd.Timestamp(FIRST_DECISION)
    wealth = {}
    for t in cal[cal <= pd.Timestamp(PAPER_END)]:
        p_g, p_b = float(gold[t]), float(btc[t])
        W = C + G * p_g + B * p_b
        if t < first_dec:
            wealth[t] = W
            continue
        if gold_traded[t]:
            tgt = w_fix
            G_new, B_new = tgt[1] * W / p_g, tgt[2] * W / p_b
            base = W
        else:
            Ws = C + B * p_b
            s = w_fix[0] + w_fix[2]
            tgt = np.array([w_fix[0] / s if s else 1.0, 0.0, w_fix[2] / s if s else 0.0])
            G_new, B_new = G, tgt[2] * Ws / p_b
            base = Ws
        cost = cost_gold * abs(G_new - G) * p_g + cost_btc * abs(B_new - B) * p_b
        C, G, B = tgt[0] * base - cost, G_new, B_new
        wealth[t] = C + G * p_g + B * p_b
    return pd.Series(wealth).sort_index()


def stats(w):
    r = w.pct_change().dropna()
    yrs = len(r) / 365
    mdd = float((w / w.cummax() - 1).min() * 100)
    return dict(final=round(float(w.iloc[-1]), 1),
                mult=round(float(w.iloc[-1] / 1000), 2),
                cagr_pct=round(((w.iloc[-1] / 1000) ** (1 / yrs) - 1) * 100, 1),
                mdd_pct=round(mdd, 1))

In [ ]:
print("Fetching the paper's data sources (LBMA + Blockchain.com)…")
panel = load_panel()
print(f"gold fixes {int(panel['gold_traded'].sum())} · BTC "
      f"{panel['btc'][START]:.2f} -> {panel['btc'][PAPER_END]:.2f} "
      f"({panel['btc'][PAPER_END]/panel['btc'][START]:.1f}x buy-and-hold)")

print("\nBuilding honest ARIMA(2,1,2) forecasts (re-fit daily, past-only — ~2 min)…")
fc = build_forecasts(panel)

print("\nRunning the stated rule + benchmarks…")
stated = run_strategy(panel, fc)
ew = run_constant_mix(panel, np.array([1 / 3, 1 / 3, 1 / 3]))
btc_bh = run_constant_mix(panel, np.array([0.0, 0.0, 1.0]))
foresight = run_strategy(panel, fc, foresight=True)          # deliberate look-ahead

rows = {
    "CLAIMED (paper)": dict(final=CLAIMED["final"], mult=136.22, cagr_pct=167.2,
                            mdd_pct=CLAIMED["mdd_pct"]),
    "Stated rule, honest replication": stats(stated),
    "Plain 1/3 1/3 1/3 daily rebalance": stats(ew),
    "Bitcoin buy & hold": stats(btc_bh),
    "Look-ahead cheat (tomorrow's prices!)": stats(foresight),
}
print("\n=== claimed -> measured ($1,000 start, 2016-09-11 -> 2021-09-10) ===")
print(f"{'':40}{'final $':>12}{'mult':>9}{'CAGR%':>8}{'maxDD%':>8}")
for k, v in rows.items():
    print(f"{k:40}{v['final']:>12,.1f}{v['mult']:>9.2f}{v['cagr_pct']:>8.1f}{v['mdd_pct']:>8.1f}")

print("""
The claim is arithmetically impossible under the paper's own rule: the 70%
equal-weight anchor caps Bitcoin at 53% while Bitcoin alone did ~74x — and even
handing the optimizer tomorrow's exact prices (the look-ahead row) lands far
short of the claimed $136,216. Full C0-C7 gate suite & report:
https://market-graphs.com/praxis/papers/360760
""")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for label, w in [("Stated rule (honest)", stated), ("Plain 1/3 mix", ew),
                 ("BTC buy & hold", btc_bh), ("Look-ahead cheat", foresight)]:
    ax.plot(w.index, w.values, label=label, lw=1.2)
ax.axhline(CLAIMED["final"], color="crimson", ls="--", lw=1,
           label="Paper's claimed $136,216")
ax.set_yscale("log"); ax.set_ylabel("Wealth ($, log scale)")
ax.set_title("Paper 360760 — claimed wealth vs what the stated rule actually does")
ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()